In [16]:
# 1. Imports
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain.text_splitter import CharacterTextSplitter

import os
from dotenv import load_dotenv
load_dotenv()

# 2. Setup
llm = ChatOpenAI(temperature=0, model="gpt-3.5-turbo")

# 3. Load document and split
with open("sample.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
docs = splitter.create_documents([raw_text])

# 4. Define prompts
map_prompt = PromptTemplate.from_template("Summarize:\n\n{context}")
reduce_prompt = PromptTemplate.from_template("Combine the following summaries into a single answer:\n\n{context}")

# 5. Create map + reduce chains
map_chain = map_prompt | llm
reduce_chain = reduce_prompt | llm

# 6. Use RunnableLambda to convert documents
def map_docs(docs):
    return [{"context": doc.page_content} for doc in docs]

def extract_texts(responses):
    return {"context": "\n".join([res.content for res in responses])}

# 7. Combine using RunnableParallel (MapReduce style)
map_reduce_chain = (
    RunnableLambda(map_docs)
    # .map runs multiple invoke commands based on the number of input docs, it takes list of dict as input and provides list of summary in output
    | map_chain.map() 
    #all the individual summaries are combined together in one single dictionary
    | RunnableLambda(extract_texts)
    # the one single dictionary of all the individual summary is then summarised to final summary
    | reduce_chain
)

# 8. Execute
result = map_reduce_chain.invoke(docs)
print("📄 Final Summary:\n", result)

📄 Final Summary:
 content='LangChain is a framework developed by Harrison Chase for building applications with LLMs. It supports various features such as RAG, agents, memory, tools, and more, and is commonly used in chatbots, document Q&A, and AI workflow applications.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 68, 'total_tokens': 120, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DE90zMw2GWSU18LHKzKDz8tCKXrd1', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--901202c4-af2f-458b-b9e7-8f1542584c8f-0' usage_metadata={'input_tokens': 68, 'output_tokens': 52, 'total_tokens': 120, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details'

In [2]:
docs

[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow')]

In [3]:
raw_text="""LangChain is a framework for building applications with LLMs.
LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.
    It’s commonly used in chatbots, document Q&A, and AI workflow"""

splitter = CharacterTextSplitter(separator="\n", chunk_size=300, chunk_overlap=50)
texts = splitter.split_text(raw_text)
print(texts)
documents = [Document(page_content=t) for t in texts]
print(documents)

['LangChain is a framework for building applications with LLMs.\nLangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.\n    It’s commonly used in chatbots, document Q&A, and AI workflow']
[Document(metadata={}, page_content='LangChain is a framework for building applications with LLMs.\nLangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.\n    It’s commonly used in chatbots, document Q&A, and AI workflow')]


In [4]:
map_chain = map_prompt | llm
map_chain

PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='Summarize:\n\n{context}')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000022330998E80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000022333642350>, root_client=<openai.OpenAI object at 0x00000223330CEC20>, root_async_client=<openai.AsyncOpenAI object at 0x00000223336422C0>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********'))

In [5]:
map_chain.map()

RunnableEach(bound=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template='Summarize:\n\n{context}')
| ChatOpenAI(client=<openai.resources.chat.completions.completions.Completions object at 0x0000022330998E80>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x0000022333642350>, root_client=<openai.OpenAI object at 0x00000223330CEC20>, root_async_client=<openai.AsyncOpenAI object at 0x00000223336422C0>, temperature=0.0, model_kwargs={}, openai_api_key=SecretStr('**********')))

In [14]:
map_docs(docs)[0]#.get('context')

{'context': 'LangChain is a framework for building applications with LLMs.LangChain was created by Harrison Chase.LangChain supports RAG, agents, memory, tools, and more.It’s commonly used in chatbots, document Q&A, and AI workflow'}

In [10]:
map_chain.invoke({"context":map_docs(docs)[0]})

AIMessage(content='LangChain is a framework created by Harrison Chase for building applications with LLMs. It supports RAG, agents, memory, tools, and is commonly used in chatbots, document Q&A, and AI workflow.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 44, 'prompt_tokens': 64, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DE8nEIXOUy6Z08gnSzVwA0l2Mg4LO', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--266bdd29-301f-4568-8f8a-db6307c297bf-0', usage_metadata={'input_tokens': 64, 'output_tokens': 44, 'total_tokens': 108, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
map_chain.map({"context":map_docs(docs)[0].get('context')})

TypeError: Runnable.map() takes 1 positional argument but 2 were given